In [ ]:
import pandas as pd
import numpy as np

from datetime import datetime

import requests

from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

In [ ]:
class Portfolio:
    def __init__(self, balance, portfolio_df, expected_annual_return, annual_volatility, sharpe_ratio, leftover):
        self.balance = "$" + str(round(balance, 2))
        self.leftover = "$" + str(round(leftover, 2))
        self.amount_used = "$" + str(round(balance-leftover, 2))
                                  
        self.portfolio_df = portfolio_df
                                  
        self.expected_annual_return = str(round(expected_annual_return, 3)*100) + "%"
        self.annual_volatility = str(round(annual_volatility, 3)*100) + "%"
        self.sharpe_ratio = str(round(sharpe_ratio, 3))
    
    def print_stats(self):
        print("Balance:", self.balance, "| Used:", self.amount_used, "| Left Over:", self.leftover)
        print("Expected Annual Return:", self.expected_annual_return)
        print("Annual Volatility / Risk:", self.annual_volatility)
        print("Sharpe Ratio:", self.sharpe_ratio)

In [ ]:
def create_data_frame(company_names, allocation, latest_prices):
    
    symbols_list = list(allocation.keys())
    symbols_shares = list(allocation.values())
    number_of_symbols = len(company_names)
    
    latest_prices = [latest_prices[symbol] for symbol in symbols_list]
    
    portfolio_df = pd.DataFrame(index=list(np.arange(number_of_symbols)),
                                columns=['Company Ticker', "Company Name", "Number Of Shares", "Latest Price"])

    portfolio_df['Company Ticker'] = symbols_list
    portfolio_df['Company Name'] = company_names
    portfolio_df['Number Of Shares'] = symbols_shares
    portfolio_df['Latest Price'] = latest_prices
    
    return portfolio_df

In [ ]:
def get_allocation_and_leftover(symbols_df, cleaned_weights, portfolio_value):
    
    latest_prices = get_latest_prices(symbols_df)
    weights = cleaned_weights
    discrete_allocation = DiscreteAllocation(weights, latest_prices, total_portfolio_value=portfolio_value)
    
    allocation, leftover = discrete_allocation.greedy_portfolio()
    
    return allocation, leftover, latest_prices

In [ ]:
def get_symbol_name(symbol):
    url = "http://d.yimg.com/autoc.finance.yahoo.com/autoc?query={}&region=1&lang=en".format(symbol)

    result = requests.get(url).json()

    for x in result['ResultSet']['Result']:
        if x['symbol'] == symbol:
            return x['name']

In [ ]:
def get_optimized_portfolio(balance, symbols_df):
    
    symbols = symbols_df.columns
    
    mu = expected_returns.mean_historical_return(symbols_df) # Expected Returns 
    S = risk_models.sample_cov(symbols_df) # Sample Covariance Matrix

    # Optimize for Max Sharpe Ratio (Return / Risk Ratio That Compares Stocks)
    ef = EfficientFrontier(mu, S)
    weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()
    expected_annual_return, annual_volatility, sharpe_ratio = ef.portfolio_performance(verbose=False)
    
    # Create Portfolio DataFrame
    allocation, leftover, latest_prices = get_allocation_and_leftover(symbols_df, cleaned_weights, balance)
    company_names = [get_symbol_name(symbol) for symbol in allocation]
    
    portfolio_df = create_data_frame(company_names, allocation, latest_prices)
    
    # Create Portfolio Object
    portfolio = Portfolio(balance, portfolio_df, expected_annual_return, annual_volatility, sharpe_ratio, leftover)
    
    return portfolio

In [ ]:
def filter_price(df, min_price):
    rows_to_keep = []
    for row in df:
        for price in df[row]:
            if price < min_price:
                rows_to_keep.append(row)

    rows_to_keep = list(dict.fromkeys(rows_to_keep))
    return df.drop(rows_to_keep, axis=1)

In [ ]:
df = pd.read_csv("NYSE_Close.csv")
df = df.set_index(pd.DatetimeIndex(df['Date'].values))
df.drop(columns=['Date'], axis=1, inplace=True)

In [ ]:
df = df.iloc[-300:, ::3]
df = df.dropna(axis=1)
df = filter_price(df, 5)
df

In [ ]:
# DF Must Have The Dates as the Index and Then Each Column is a different stock

In [ ]:
portfolio = get_optimized_portfolio(2000, df)

In [ ]:
portfolio.print_stats()

In [ ]:
portfolio.portfolio_df.sort_values("Number Of Shares", ascending=False)